# 03 · Input pipeline, models and losses: checks (tasks T13, T14, T15)

Before any training, each building block is checked on its own:

1. **Input pipeline (T13)**: do images and masks still line up after cropping, flipping and rotating?
2. **Models (T14)**: do all three U-Nets build, and does a 256×256 input give a 256×256 output?
3. **Losses and the validation metric (T15)**: do they give the values worked out by hand, and do they ignore the −1 (no data) pixels?
4. **All together**: can the model memorise a single batch? If it cannot, something is broken.

Only training and validation data are used.

In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT))
import src  # noqa: F401  (must come before TensorFlow: switches on the XLA compile cache)

import json

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from src import viz
from src.losses import LOSSES, WaterIoU, bce_dice_loss, bce_loss, dice_loss
from src.models import ENCODERS, ENCODER_LABELS, MOBILENET_SKIPS, build_unet, count_parameters
from src.pipeline import BATCH, CROP, CROPS_PER_CHIP, load_arrays, make_eval_dataset, make_train_dataset, steps_per_epoch
from src.preprocess import CHANNEL_SETS, channel_names
from src.train import setup_runtime

viz.use_style()
setup_runtime(mixed_precision=True)
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 1. Input pipeline (T13)

Per chip: read VV, VH and the label → clip to [−35, 5] dB → build the channels → standardise with the training statistics → NaN becomes 0. The label keeps −1 so the loss can ignore those pixels.
All chips of a split are held in memory; `tf.data` is built on top of the arrays.

In [ ]:
X, Y = load_arrays("train", "vv_vh_ratio")
print("training inputs :", X.shape, X.dtype, "| NaN left:", int(np.isnan(X).sum()))
print("training labels :", Y.shape, Y.dtype, "| values:", np.unique(Y).tolist())
print("channels        :", channel_names("vv_vh_ratio"))
for name in CHANNEL_SETS:   # the channel set is a setting: E1 switches inputs without new code
    print(f"  channel set {name!r:15s} -> input shape {load_arrays('train', name)[0].shape}")

In [ ]:
train_ds = make_train_dataset(X, Y, seed=0)
images, masks = next(iter(train_ds))
images, masks = images.numpy(), masks.numpy()
n_batches = sum(1 for _ in train_ds)
print("one training batch :", images.shape, masks.shape, "| mask values:", np.unique(masks).tolist())
print(f"batches per epoch  : {n_batches}  (= {len(X)} chips x {CROPS_PER_CHIP} crops / batch {BATCH} = "
      f"{steps_per_epoch(len(X))})")

Xv, Yv = load_arrays("val", "vv_vh_ratio")
val_images, val_masks = next(iter(make_eval_dataset(Xv, Yv)))
print("one validation batch:", val_images.shape, val_masks.shape, "(full chips, no augmentation)")

again, _ = next(iter(make_train_dataset(X, Y, seed=0)))
print("same seed gives the same crops:", bool(np.array_equal(images, again.numpy())))

**Do image and mask line up?** If the same crop, flip and rotation hit both, water in the mask must sit on the dark areas of VH.

In [ ]:
vh = images[..., 1]
water, land = masks[..., 0] == 1, masks[..., 0] == 0
print(f"mean standardised VH under 'water' pixels    : {vh[water].mean():+.2f}")
print(f"mean standardised VH under 'not water' pixels: {vh[land].mean():+.2f}   (water must be clearly lower)")
assert vh[water].mean() < vh[land].mean() - 0.5

order = np.argsort(-(masks[..., 0] == 1).mean(axis=(1, 2)))[:6]   # the crops of this batch with the most water
fig, axes = plt.subplots(2, 6, figsize=(10.5, 3.9))
for col, i in enumerate(order):
    axes[0, col].imshow(vh[i], cmap="gray", vmin=-3, vmax=3, interpolation="nearest")
    viz._image_axis(axes[0, col], f"crop {i}")
    viz.show_label(axes[1, col], masks[i, ..., 0])
axes[0, 0].set_ylabel("VH (standardised)", fontsize=8, color=viz.INK_2)
axes[1, 0].set_ylabel("mask", fontsize=8, color=viz.INK_2)
fig.legend(handles=viz.label_legend_handles(), loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.04))
fig.tight_layout(h_pad=0.4, w_pad=0.3)
viz.savefig(fig, ROOT / "figures" / "pipeline_batch.png")
plt.show()

The blue areas of the masks sit on the dark areas of VH in every crop, including flipped and rotated ones: the augmentation keeps image and mask together. (`tests/test_pipeline.py` checks the same thing exactly, on an image whose pixels carry their own index.)

## 2. Models (T14)

`build_unet(encoder, in_channels)` with three encoders. All end in a single sigmoid channel: the probability of water.

In [ ]:
counts, rows = {}, []
for encoder in ENCODERS:
    model = build_unet(encoder, 3)
    out_256 = model(np.zeros((1, 256, 256, 3), dtype="float32"))
    out_512 = model(np.zeros((1, 512, 512, 3), dtype="float32"))
    assert out_256.shape == (1, 256, 256, 1) and out_512.shape == (1, 512, 512, 1)
    counts[encoder] = count_parameters(model)
    decoder = sum(l.count_params() for l in model.layers if l.name.startswith(("dec", "water_prob")))
    rows.append({"model": ENCODER_LABELS[encoder], "parameters": counts[encoder]["total_params"],
                 "trainable": counts[encoder]["trainable_params"],
                 "of which decoder": decoder if encoder != "scratch" else None,
                 "output for (1, 256, 256, 3)": tuple(out_256.shape), "output dtype": str(out_256.dtype)})
(ROOT / "results" / "model_params.json").write_text(json.dumps(counts, indent=2) + "\n")
pd.DataFrame(rows).set_index("model")

In [ ]:
mnet = build_unet("mobilenetv2", 3)
skips = pd.DataFrame([{"layer": n, "output shape for a 256x256 input": mnet.get_layer(n).output.shape[1:],
                       "scale": f"1/{256 // 128 * 2 ** i}"} for i, n in enumerate(MOBILENET_SKIPS)])
print("MobileNetV2 feature maps used by the decoder (the last one is the bottleneck):")
skips["output shape for a 256x256 input"] = [tuple(256 // (2 ** (i + 1)) if d is None else d for d in s)
                                             for i, s in enumerate(skips["output shape for a 256x256 input"])]
skips

In [ ]:
for channels in (1, 2):   # VH only and VV+VH need a first layer with 1 or 2 input channels
    for encoder in ("scratch", "mobilenetv2"):
        assert build_unet(encoder, channels)(np.zeros((1, 256, 256, channels), dtype="float32")).shape == (1, 256, 256, 1)
try:
    build_unet("mobilenetv2_imagenet", 2)
except ValueError as e:
    print("as intended:", e)
bn = [l.momentum for l in mnet.layers if isinstance(l, keras.layers.BatchNormalization)]
print(f"{len(bn)} BatchNorm layers in the MobileNetV2 U-Net, momentum values in use: {sorted(set(bn))}")
print("layer compute dtype:", mnet.get_layer("dec1_conv1").compute_dtype, "| output dtype:", mnet.output.dtype)

* All three models build and keep the spatial size, for 256×256 crops and for full 512×512 chips, with 1, 2 or 3 input channels (ImageNet weights require exactly 3).
* The scratch U-Net is the smallest model. In the MobileNetV2 U-Nets more than half of the parameters sit in the decoder, mostly in the first decoder block that fuses the two deepest feature maps.
* Under mixed precision the layers compute in float16 but the output layer is float32, so the sigmoid and the losses are computed at full precision.

## 3. Losses and the validation metric (T15)

A 3×3 example small enough to work out by hand. Two pixels are no-data (−1); the predictions on them (0.9 and 0.5) must not matter.

In [ ]:
label = np.array([[1, 1, 0], [0, -1, 1], [0, 0, -1]], dtype="float32")
pred = np.array([[0.9, 0.2, 0.8], [0.1, 0.9, 0.7], [0.9, 0.2, 0.5]], dtype="float32")
t = lambda a: tf.constant(a[None, :, :, None])

by_hand = {
    "BCE": -(np.log(.9) + np.log(.2) + np.log(.7) + np.log(1 - .8) + np.log(1 - .1) + np.log(1 - .9) + np.log(1 - .2)) / 7,
    "Dice": 1 - (2 * (.9 + .2 + .7)) / ((.9 + .2 + .8 + .1 + .7 + .9 + .2) + 3),
}
by_hand["BCE+Dice"] = by_hand["BCE"] + by_hand["Dice"]
by_hand["water IoU at 0.5"] = 2 / 5   # TP 2, FP 2, FN 1 on the 7 valid pixels

metric = WaterIoU()
metric.update_state(t(label), t(pred))
computed = {"BCE": float(bce_loss(t(label), t(pred))), "Dice": float(dice_loss(t(label), t(pred))),
            "BCE+Dice": float(bce_dice_loss(t(label), t(pred))), "water IoU at 0.5": float(metric.result())}

changed = pred.copy()
changed[1, 1], changed[2, 2] = 0.0, 1.0     # change the predictions on the two no-data pixels
metric2 = WaterIoU()
metric2.update_state(t(label), t(changed))
after = {"BCE": float(bce_loss(t(label), t(changed))), "Dice": float(dice_loss(t(label), t(changed))),
         "BCE+Dice": float(bce_dice_loss(t(label), t(changed))), "water IoU at 0.5": float(metric2.result())}

perfect = np.where(label == 1, 1.0, 0.0).astype("float32")
table = pd.DataFrame({"worked out by hand": by_hand, "computed": computed, "computed after changing the no-data pixels": after})
assert np.allclose(table["worked out by hand"], table["computed"], rtol=1e-4)
assert np.allclose(table["computed"], table["computed after changing the no-data pixels"], rtol=1e-6)
print(f"Dice loss of a perfect prediction: {float(dice_loss(t(label), t(perfect))):.2e}")
table

In [ ]:
done = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests"], cwd=ROOT,
                      capture_output=True, text=True)
print(done.stdout.strip().splitlines()[-1])
assert done.returncode == 0, done.stdout[-3000:]

The losses and the metric give the hand-calculated values, a perfect prediction gives a Dice loss of about 0, and changing the prediction on a −1 pixel changes nothing. The full test suite (`tests/`: metrics, losses, pipeline, models, configs) passes.

## 4. All together: can the model memorise one batch?

A standard sanity check. The default model is trained on one fixed batch for 80 steps, through the same `compile` / training-step code that the real runs use (Adam, BCE + Dice, mixed precision). With everything wired correctly the loss must fall steeply and the IoU on that batch must climb towards 1.
The numbers are measured in training mode, i.e. BatchNorm uses the batch's own statistics, which is what "memorise this batch" means.

In [ ]:
keras.utils.set_random_seed(0)
model = build_unet("mobilenetv2", 3)
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=bce_dice_loss, metrics=[WaterIoU()], jit_compile=False)
print(f"the batch: {(masks == 1).mean():.1%} water, {(masks == -1).mean():.1%} no data")

log = []
for i in range(80):
    model.reset_metrics()                       # report this step alone, not a running average
    loss, iou = model.train_on_batch(images, masks)
    log.append({"step": i + 1, "loss": float(loss), "water IoU on the batch": float(iou)})
log = pd.DataFrame(log).set_index("step")
assert log["loss"].iloc[-1] < 0.5 * log["loss"].iloc[0] and log["water IoU on the batch"].iloc[-1] > 0.85
log.iloc[[0, 4, 9, 19, 39, 59, 79]].round(4)

The loss collapses and the IoU on the memorised batch climbs towards 1: data, model, loss and optimiser are wired together correctly. The building blocks are ready for the smoke test (notebook 04).